# 01 - Understand and prepare the data
**Question:** What does one row represent, and what information can we use before an hour begins?

One row records the total rentals during an observed hour in Washington, D.C., in 2011-2012.
The target `cnt` equals `casual + registered`. We predict total rentals, so neither component is an input.

Run each cell in order. This notebook verifies the original CSV, describes TRAINING data with SQL,
builds two historical predictors, and writes three time partitions. No earlier notebook is required.


In [ ]:
from pathlib import Path
import sys
# Support opening the notebook from the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'mobility').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'mobility').is_dir():
    raise RuntimeError('Open Jupyter from the project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mobility.data import OUTPUT, invalidate_outputs
import pandas as pd
import numpy as np
import json
from IPython.display import display

from mobility.data import load_data
from mobility.features import FEATURES, make_features, split_by_time
invalidate_outputs(1)
raw = load_data()
print(f'{len(raw):,} observed hours; {raw.timestamp.min()} to {raw.timestamp.max()}')
display(raw.head())


## 1. Check what is missing
The file has no missing cells, but some clock hours have no row. We do not assume those hours had zero rentals.
A checksum checks the file bytes against `data/source.json`; validation also checks counts, ranges,
unique timestamps, the target identity, and calendar consistency. A checksum alone does not prove data quality.


In [ ]:
all_hours = pd.date_range(raw.timestamp.min(), raw.timestamp.max(), freq='h')
missing = all_hours.difference(raw.timestamp)
print('Missing clock hours:', len(missing))
print('Missing cells:', int(raw.isna().sum().sum()))
print('Target identity holds:', bool((raw.casual + raw.registered == raw.cnt).all()))
print('First missing timestamps:', list(missing[:5]))


## 2. Describe training data with SQL and pandas
Use only dates before July 2012 to explore patterns before model selection. The SQL groups by hour and
working-day status. `COUNT` counts observed hours, `SUM` totals rentals, and `AVG` describes typical volume.
The database is a local teaching artifact, not a production data warehouse.


In [ ]:
import sqlite3
training_raw = raw[raw.timestamp < '2012-07-01'].copy()
query = (ROOT / 'sql' / 'hourly_summary.sql').read_text(encoding='utf-8')
with sqlite3.connect(OUTPUT / 'hourly.db') as connection:
    training_raw.to_sql('hourly', connection, if_exists='replace', index=False)
    hourly = pd.read_sql_query(query, connection)
pandas_hourly = (training_raw.groupby(['workingday', 'hr'])['cnt']
                 .agg(observed_hours='count', total_rentals='sum', mean_rentals='mean')
                 .reset_index())
pd.testing.assert_frame_equal(hourly, pandas_hourly, check_dtype=False)
hourly.to_csv(OUTPUT / 'hourly_summary.csv', index=False)
display(hourly.head())
print('SQL and pandas agree on every group.')


In [ ]:
import matplotlib.pyplot as plt
curve = hourly.pivot(index='hr', columns='workingday', values='mean_rentals')
curve.columns = ['Weekend / holiday', 'Working day']
ax = curve.plot(figsize=(9, 4), marker='o')
ax.set(xlabel='Hour of day', ylabel='Mean observed rentals', title='Training period: rentals by hour')
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.savefig(OUTPUT / 'hourly_demand.png', dpi=150)
plt.show()


## 3. Build features using the actual clock
`lag_24` is the recorded rentals at the exact timestamp 24 hours earlier; `lag_168` is the same hour
one week earlier. We use timestamp lookup, because shifting 24 rows can cross missing clock hours.
We drop a row if either lag is missing. No future averages or invented counts fill the gaps.

The eight model inputs are six calendar fields and these two lags. Observed weather, record IDs,
and the target components are excluded. This keeps the prediction inputs available before the hour.

**Forecast setup:** one hour at a time with historical rentals already recorded. Earlier observations
from the validation/test period can supply lags for later hours. This is not a forecast of an entire
quarter issued on its first day.


In [ ]:
features = make_features(raw)
train, validation, test = split_by_time(features)
partitions = {'train': train, 'validation': validation, 'test': test}
report = {'raw_rows': len(raw), 'missing_clock_hours': len(missing),
          'eligible_rows': len(features), 'excluded_for_missing_history': len(raw)-len(features),
          'features': FEATURES, 'partitions': {}}
for name, frame in partitions.items():
    frame.to_csv(OUTPUT / f'{name}.csv', index=False)
    report['partitions'][name] = {'rows': len(frame), 'start': str(frame.timestamp.min()),
                                 'end': str(frame.timestamp.max())}
(OUTPUT / 'data_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
display(pd.DataFrame(report['partitions']).T)
print('Eligible:', len(features), '| Excluded for missing history:', len(raw)-len(features))
print('Predictors:', FEATURES)
display(features[['timestamp', 'lag_24', 'lag_168', 'cnt']].head())


## What you should be able to explain
- Why `casual` and `registered` would reveal the answer.
- Why a missing row differs from zero rentals.
- Why a lag uses clock time and only past observations.
- Train ends June 2012; validation covers July-September; test covers October-December.

**Next:** open `02_model.ipynb`. The three CSVs and `data_report.json` are the handoff.
